<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab05.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 5 — CAPSTONE: The Ground-State Energy of H$_2$ with VQE

**Maps to:** Module 3 (whole) + Module 4, Lesson 4 (Reduced Mapping in H$_2$)

**Time:** ~75 minutes (instructor walkthrough ~15 min)

---

This is the lab everything else was building toward. You will go end to end:

```
molecule geometry  ->  Hamiltonian  ->  ansatz circuit  ->  measure E(theta)
                                              ^                    |
                          inner loop:  classical optimizer  <------+
                                              |
                       outer loop:  change R, repeat  ->  dissociation curve
```

and land on the two numbers from the Module 3 slides: **$R_{eq} \approx 0.74$ Å** and
**$E_0 \approx -1.1373$ Ha $= -30.9$ eV**.

### After this lab you can
1. Produce a qubit Hamiltonian for any small molecule with Qiskit Nature + PySCF.
2. Recognize the lecture's $c_0 \dots c_4$ in the output.
3. Build and reason about the reduced 2-qubit ansatz.
4. Run the inner VQE loop with COBYLA and with finite shots.
5. Run the outer geometry loop and produce the potential energy curve.
6. Quantify the *correlation energy* — the part Hartree–Fock misses, i.e. exactly the
   "tiny percentage of antibonding" from Module 3.

In [ ]:
# %pip install -q qiskit qiskit-aer qiskit-nature pyscf matplotlib scipy
import numpy as np, time
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer.primitives import EstimatorV2

np.set_printoptions(precision=5, suppress=True)
HARTREE_TO_EV = 27.2114

try:
    from qiskit_nature.units import DistanceUnit
    from qiskit_nature.second_q.drivers import PySCFDriver
    from qiskit_nature.second_q.mappers import ParityMapper
    HAVE_NATURE = True
except Exception as e:
    HAVE_NATURE = False
    print("Qiskit Nature/PySCF not available -> using the stored coefficient table.", e)
print("Qiskit Nature available:", HAVE_NATURE)

## Step 0 — Molecule in, Hamiltonian out

Three steps hide behind one function call:

1. **PySCF** solves the classical Hartree–Fock problem in the STO-3G basis and hands back
   the one- and two-electron integrals — the numbers in the $H$ formula on your
   "Rule of the Energy" slide.
2. Those integrals are written as a **fermionic** operator (creation/annihilation
   operators $a_p^\dagger a_q$).
3. A **mapper** converts fermions into qubits. `JordanWignerMapper` needs 4 qubits for
   H$_2$; `ParityMapper` plus symmetry reduction needs only **2** — the reduced mapping
   from your Module 4 slide.

In [ ]:
# Fallback table: coefficients (c0,c1,c2,c3,c4) and nuclear repulsion vs bond length.
H2_TABLE = {
    0.30: (-0.75374195, +0.80864891, -0.80864891, -0.01328798, +0.16081852, +1.76392404),
    0.35: (-0.81066178, +0.74741582, -0.74741582, -0.01310364, +0.16257322, +1.51193489),
    0.40: (-0.86257953, +0.68881943, -0.68881943, -0.01291397, +0.16451542, +1.32294303),
    0.45: (-0.90840213, +0.63388978, -0.63388978, -0.01271920, +0.16662140, +1.17594936),
    0.50: (-0.94770788, +0.58307963, -0.58307963, -0.01251643, +0.16887023, +1.05835442),
    0.55: (-0.98051390, +0.53648878, -0.53648878, -0.01230035, +0.17124452, +0.96214038),
    0.60: (-1.00712708, +0.49401379, -0.49401379, -0.01206439, +0.17373064, +0.88196202),
    0.65: (-1.02805041, +0.45543342, -0.45543342, -0.01180192, +0.17631845, +0.81411879),
    0.70: (-1.04391252, +0.42045568, -0.42045568, -0.01150740, +0.17900058, +0.75596744),
    0.71: (-1.04653791, +0.41386358, -0.41386358, -0.01144429, +0.17954781, +0.74532002),
    0.72: (-1.04899406, +0.40739954, -0.40739954, -0.01137972, +0.18009857, +0.73496835),
    0.73: (-1.05128660, +0.40106074, -0.40106074, -0.01131368, +0.18065279, +0.72490029),
    0.74: (-1.05342108, +0.39484436, -0.39484436, -0.01124616, +0.18121046, +0.71510434),
    0.75: (-1.05540303, +0.38874759, -0.38874759, -0.01117714, +0.18177154, +0.70556961),
    0.76: (-1.05723792, +0.38276759, -0.38276759, -0.01110664, +0.18233598, +0.69628580),
    0.77: (-1.05893111, +0.37690156, -0.37690156, -0.01103464, +0.18290377, +0.68724313),
    0.78: (-1.06048789, +0.37114671, -0.37114671, -0.01096115, +0.18347485, +0.67843232),
    0.79: (-1.06191343, +0.36550024, -0.36550024, -0.01088618, +0.18404920, +0.66984457),
    0.80: (-1.06321280, +0.35995942, -0.35995942, -0.01080973, +0.18462678, +0.66147151),
    0.85: (-1.06798506, +0.33374649, -0.33374649, -0.01040607, +0.18756185, +0.62256142),
    0.90: (-1.07028327, +0.30978728, -0.30978728, -0.00996911, +0.19057169, +0.58797468),
    0.95: (-1.07057706, +0.28779599, -0.28779599, -0.00950347, +0.19365032, +0.55702864),
    1.00: (-1.06924349, +0.26752865, -0.26752865, -0.00901493, +0.19679058, +0.52917721),
    1.05: (-1.06657841, +0.24878329, -0.24878329, -0.00850994, +0.19998427, +0.50397830),
    1.10: (-1.06281249, +0.23139588, -0.23139588, -0.00799518, +0.20322223, +0.48107019),
    1.15: (-1.05812757, +0.21523394, -0.21523394, -0.00747720, +0.20649467, +0.46015410),
    1.20: (-1.05267072, +0.20018958, -0.20018958, -0.00696216, +0.20979147, +0.44098101),
    1.25: (-1.04656497, +0.18617310, -0.18617310, -0.00645559, +0.21310240, +0.42334177),
    1.30: (-1.03991659, +0.17310785, -0.17310785, -0.00596229, +0.21641746, +0.40705939),
    1.35: (-1.03281973, +0.16092639, -0.16092639, -0.00548622, +0.21972704, +0.39198312),
    1.40: (-1.02535904, +0.14956794, -0.14956794, -0.00503054, +0.22302209, +0.37798372),
    1.45: (-1.01761100, +0.13897678, -0.13897678, -0.00459759, +0.22629426, +0.36494980),
    1.50: (-1.00964469, +0.12910131, -0.12910131, -0.00418896, +0.22953594, +0.35278481),
    1.55: (-1.00152211, +0.11989354, -0.11989354, -0.00380558, +0.23274029, +0.34140465),
    1.60: (-0.99329850, +0.11130875, -0.11130875, -0.00344779, +0.23590129, +0.33073576),
    1.65: (-0.98502265, +0.10330533, -0.10330533, -0.00311546, +0.23901365, +0.32071346),
    1.70: (-0.97673724, +0.09584459, -0.09584459, -0.00280807, +0.24207284, +0.31128071),
    1.75: (-0.96847929, +0.08889055, -0.08889055, -0.00252482, +0.24507502, +0.30238698),
    1.80: (-0.96028058, +0.08240979, -0.08240979, -0.00226467, +0.24801699, +0.29398734),
    1.85: (-0.95216812, +0.07637120, -0.07637120, -0.00202649, +0.25089615, +0.28604174),
    1.90: (-0.94416461, +0.07074579, -0.07074579, -0.00180902, +0.25371043, +0.27851432),
    1.95: (-0.93628889, +0.06550650, -0.06550650, -0.00161098, +0.25645825, +0.27137293),
    2.00: (-0.92855635, +0.06062801, -0.06062801, -0.00143110, +0.25913847, +0.26458861),
    2.05: (-0.92097933, +0.05608661, -0.05608661, -0.00126812, +0.26175037, +0.25813522),
    2.10: (-0.91356747, +0.05186007, -0.05186007, -0.00112080, +0.26429357, +0.25198915),
    2.15: (-0.90632806, +0.04792750, -0.04792750, -0.00098799, +0.26676799, +0.24612894),
    2.20: (-0.89926632, +0.04426934, -0.04426934, -0.00086855, +0.26917386, +0.24053510),
    2.25: (-0.89238567, +0.04086723, -0.04086723, -0.00076144, +0.27151165, +0.23518987),
    2.30: (-0.88568799, +0.03770401, -0.03770401, -0.00066564, +0.27378205, +0.23007705),
    2.35: (-0.87917385, +0.03476361, -0.03476361, -0.00058020, +0.27598597, +0.22518179),
    2.40: (-0.87284269, +0.03203106, -0.03203106, -0.00050424, +0.27812444, +0.22049050),
    2.45: (-0.86669300, +0.02949241, -0.02949241, -0.00043691, +0.28019869, +0.21599070),
    2.50: (-0.86072251, +0.02713470, -0.02713470, -0.00037742, +0.28221005, +0.21167088),
    2.55: (-0.85492827, +0.02494587, -0.02494587, -0.00032503, +0.28415994, +0.20752047),
    2.60: (-0.84930684, +0.02291473, -0.02291473, -0.00027904, +0.28604991, +0.20352970),
}

def h2_hamiltonian(R):
    '''Return (SparsePauliOp qubit Hamiltonian, nuclear repulsion energy) for H2 at
    bond length R in Angstrom, in the reduced 2-qubit encoding.'''
    if HAVE_NATURE:
        problem = PySCFDriver(atom=f"H 0 0 0; H 0 0 {R}", basis="sto3g",
                              unit=DistanceUnit.ANGSTROM).run()
        mapper = ParityMapper(num_particles=problem.num_particles)
        return mapper.map(problem.hamiltonian.second_q_op()), problem.nuclear_repulsion_energy
    key = round(R, 2)
    c0, c1, c2, c3, c4, enuc = H2_TABLE[key]
    return SparsePauliOp.from_list([("II", c0), ("IZ", c1), ("ZI", c2),
                                    ("ZZ", c3), ("XX", c4)]), enuc

H, E_nuc = h2_hamiltonian(0.74)
print(H)
print(f"\nnuclear repulsion at R = 0.74 A : {E_nuc:.6f} Ha")

Compare with the coefficients on the Module 3 slide (quoted at $R = 0.735$ Å):

```
c0 = -1.0523732458      c1 = +0.3979374248      c2 = -0.3979374248
c3 = -0.0112801043      c4 = +0.1809311998
```

### Exercise 1 — confirm the lecture's numbers

In [ ]:
H735, enuc735 = h2_hamiltonian(0.735 if HAVE_NATURE else 0.70)
coeffs = {p.to_label(): c.real for p, c in zip(H735.paulis, H735.coeffs)}
for k, v in coeffs.items():
    print(f"  {k}: {v:+.10f}")

if HAVE_NATURE:
    assert np.isclose(coeffs["II"], -1.0523732458, atol=1e-6)
    assert np.isclose(coeffs["IZ"], +0.3979374248, atol=1e-6)
    assert np.isclose(coeffs["XX"], +0.1809311998, atol=1e-6)
    print("\nPASS -- these are exactly the lecture's c0, c1, c2, c3, c4.")

E_exact_elec = np.linalg.eigvalsh(H735.to_matrix())[0]
print(f"\nexact (diagonalized) ground state: {E_exact_elec + enuc735:+.6f} Ha "
      f"= {(E_exact_elec + enuc735)*HARTREE_TO_EV:.2f} eV")

Diagonalizing a 4×4 matrix is of course trivial classically. The point of the rest of the
lab is that this route — prepare a state, *measure* its energy, let a classical optimizer
turn the knob — is the one that keeps working when the matrix is $2^{100}\times 2^{100}$.

## Step 1 — The ansatz

The reduced mapping uses two qubits with this meaning:

| state | meaning |
|---|---|
| $\lvert01\rangle$ | both electrons in the **bonding** orbital — the Hartree–Fock reference |
| $\lvert10\rangle$ | both electrons in the **antibonding** orbital |
| $\lvert00\rangle,\lvert11\rangle$ | unphysical here |

So the ansatz must (a) start at $|01\rangle$ and (b) mix in a controllable amount of
$|10\rangle$ — precisely the mixer of Lab 3, in its real-amplitude Givens form.

### Exercise 2 — build it and check the two limits

In [ ]:
def h2_ansatz(theta):
    qc = QuantumCircuit(2)
    qc.x(0)                                   # HF reference |01>
    qc.s(1); qc.h(1); qc.h(0)
    # TODO: the entangler core -- cx(0,1), rz(2*theta, 1), cx(0,1)
    ...
    qc.h(1); qc.sdg(1); qc.h(0)
    return qc

print(h2_ansatz(0.112).draw(output="text"))

for t in [0.0, 0.112, np.pi/4]:
    p = Statevector(h2_ansatz(t)).probabilities_dict()
    print(f"theta={t:6.3f}  P(01)={p.get('01',0):.4f}  P(10)={p.get('10',0):.4f}  "
          f"P(00)+P(11)={p.get('00',0)+p.get('11',0):.2e}")

p0 = Statevector(h2_ansatz(0.0)).probabilities_dict()
assert np.isclose(p0.get("01", 0), 1.0, atol=1e-9), "theta=0 must be the HF state"
print("\nPASS")

## Step 2 — The energy landscape

Before letting an optimizer loose, look at what it has to walk down. We use Qiskit's
**Estimator** primitive, which takes a circuit and an observable and returns
$\langle\psi|H|\psi\rangle$ directly. Setting `default_precision=0.0` means "no shot
noise" — the exact value. (Step 4 turns the noise back on.)

In [ ]:
estimator_exact = EstimatorV2(options={"default_precision": 0.0})

def energy(theta, H, E_nuc, estimator=estimator_exact):
    '''Total energy of the ansatz at angle theta.'''
    result = estimator.run([(h2_ansatz(float(theta)), H)]).result()
    return float(result[0].data.evs) + E_nuc

H, E_nuc = h2_hamiltonian(0.74)
thetas = np.linspace(-np.pi/2, np.pi/2, 200)
Es = [energy(t, H, E_nuc) for t in thetas]

i = int(np.argmin(Es))
print(f"grid minimum: theta* = {thetas[i]:+.4f} rad,  E = {Es[i]:.6f} Ha "
      f"= {Es[i]*HARTREE_TO_EV:.2f} eV")
print(f"Hartree-Fock (theta = 0): E = {energy(0.0, H, E_nuc):.6f} Ha")

plt.figure(figsize=(6.4, 3.6))
plt.plot(thetas, Es, lw=1.6)
plt.plot(thetas[i], Es[i], "ro", label=f"minimum, $\\theta^*$={thetas[i]:.3f}")
plt.plot(0, energy(0.0, H, E_nuc), "ks", label="Hartree-Fock ($\\theta$=0)")
plt.xlabel(r"$\theta$"); plt.ylabel("total energy (Ha)")
plt.title("H$_2$ energy landscape at R = 0.74 A")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

### Exercise 3 — read the physics off the plot

Answer in the cell below (as comments or `print`s):

* How much lower is the minimum than the $\theta=0$ (Hartree–Fock) point, in mHa and in
  kcal/mol? (1 Ha = 627.5 kcal/mol.)
* What fraction of the state is antibonding at $\theta^*$?
* Is that fraction consistent with "a tiny percentage of destructive configuration"?

In [ ]:
theta_star = thetas[i]
E_hf  = ...        # TODO: energy at theta = 0
E_vqe = Es[i]
corr  = ...        # TODO: the difference
frac_anti = ...    # TODO: P(|10>) at theta_star  (Statevector(...).probabilities_dict())

print(f"correlation energy   = {corr*1000:.2f} mHa = {corr*627.5:.2f} kcal/mol")
print(f"antibonding fraction = {100*frac_anti:.2f} %")

## Step 3 — The inner loop (classical optimizer)

A grid scan is only possible because there is **one** parameter. Real ansätze have
hundreds, so we hand the job to a classical optimizer. COBYLA is the standard choice for
VQE: gradient-free, robust to a little noise, few function evaluations.

### Exercise 4 — run VQE and count the calls

In [ ]:
def run_vqe(H, E_nuc, theta0=0.0, estimator=estimator_exact, maxiter=200, record=None):
    calls = {"n": 0}
    def objective(x):
        calls["n"] += 1
        e = energy(x[0], H, E_nuc, estimator)
        if record is not None:
            record.append(e)
        return e
    res = minimize(objective, [theta0], method="COBYLA",
                   options={"maxiter": maxiter, "rhobeg": 0.3})
    return res.x[0], res.fun, calls["n"]

history = []
theta_opt, E_opt, n_calls = run_vqe(H, E_nuc, record=history)
print(f"VQE:   theta* = {theta_opt:+.6f}   E = {E_opt:.8f} Ha   ({n_calls} energy evaluations)")
print(f"grid:  theta* = {theta_star:+.6f}   E = {E_vqe:.8f} Ha   (200 evaluations)")
print(f"exact: {np.linalg.eigvalsh(H.to_matrix())[0] + E_nuc:.8f} Ha")
assert abs(E_opt - (np.linalg.eigvalsh(H.to_matrix())[0] + E_nuc)) < 1e-6

plt.figure(figsize=(5.6, 3.2))
plt.plot(history, "o-", ms=3)
plt.xlabel("energy evaluation"); plt.ylabel("E (Ha)"); plt.title("COBYLA rolling downhill")
plt.tight_layout(); plt.show()

## Step 4 — With shots (what hardware actually gives you)

Set a finite `default_precision` and the Estimator samples instead of computing exactly.
Watch what happens to the optimizer.

In [ ]:
estimator_shots = EstimatorV2(options={"default_precision": 0.01, "run_options": {"seed": 42}})

print(f"{'run':>4} {'theta*':>10} {'E (Ha)':>12} {'error (mHa)':>13}")
E_ref = np.linalg.eigvalsh(H.to_matrix())[0] + E_nuc
for k in range(5):
    est = EstimatorV2(options={"default_precision": 0.01, "run_options": {"seed": 100 + k}})
    th, e, n = run_vqe(H, E_nuc, estimator=est, maxiter=60)
    print(f"{k:>4} {th:>10.4f} {e:>12.5f} {1000*(e-E_ref):>13.2f}")

print("\nThe optimizer now stops at a slightly different theta every time, and the")
print("reported energy can even dip BELOW the true ground state -- that is noise,")
print("not physics. In production you re-evaluate the final theta with many more shots.")

### Exercise 5 — noise floor vs precision

Re-run the shot-based VQE at three precision settings and report the spread of the final
energy. Then answer: which is more wasteful, tightening precision during the search, or
running a cheap search followed by one expensive final evaluation?

In [ ]:
for prec in [0.05, 0.01, 0.002]:
    finals = []
    for k in range(5):
        # TODO: build an EstimatorV2 with this default_precision, run VQE, collect e
        ...
    finals = np.array(finals)
    print(f"precision={prec:<6} mean E = {finals.mean():+.5f} Ha   "
          f"spread = {1000*finals.std():5.2f} mHa")

## Step 5 — The outer loop: the dissociation curve

Everything so far was at one fixed bond length. The **outer loop** changes the geometry,
rebuilds the Hamiltonian, and reruns the inner loop. For H$_2$ there is exactly one
geometry parameter, $R$; for H$_2$O there are two ($r$ and $\alpha$); for a protein there
are thousands.

Runtime note: ~20 seconds.

In [ ]:
t0 = time.time()
Rs = np.round(np.arange(0.30, 2.601, 0.05), 2)
E_vqe_curve, E_hf_curve, E_exact_curve = [], [], []

for R in Rs:
    Hr, enuc = h2_hamiltonian(R)
    th, e, _ = run_vqe(Hr, enuc, theta0=0.0)
    E_vqe_curve.append(e)
    E_hf_curve.append(energy(0.0, Hr, enuc))
    E_exact_curve.append(np.linalg.eigvalsh(Hr.to_matrix())[0] + enuc)

E_vqe_curve = np.array(E_vqe_curve); E_hf_curve = np.array(E_hf_curve)
E_exact_curve = np.array(E_exact_curve)
print(f"done in {time.time()-t0:.1f} s;  max |VQE - exact| = "
      f"{1000*np.max(np.abs(E_vqe_curve - E_exact_curve)):.4f} mHa")

# Refine: the coarse grid only locates R_eq to +/- 0.05 A. Rescan finely around it.
j_coarse = int(np.argmin(E_vqe_curve))
R_fine = np.round(np.arange(max(0.70, Rs[j_coarse]-0.04), Rs[j_coarse]+0.041, 0.01), 2)
E_fine = [run_vqe(*h2_hamiltonian(R))[1] for R in R_fine]
print("fine scan:", dict(zip(R_fine, np.round(E_fine, 6))))

In [ ]:
R_eq, E_min = R_fine[int(np.argmin(E_fine))], min(E_fine)
print(f"equilibrium bond length R_eq = {R_eq:.2f} A     (experiment 0.741 A; STO-3G predicts 0.735)")
print(f"ground state energy     E_0  = {E_min:.5f} Ha = {E_min*HARTREE_TO_EV:.2f} eV")
print(f"                             (Module 3 slide: -30.9 eV)")
print(f"dissociation energy     D_e  = {(E_vqe_curve[-1]-E_min)*HARTREE_TO_EV:.2f} eV "
      f"(experiment ~4.75 eV; a minimal basis is not quantitative)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(Rs, E_hf_curve, "s--", ms=3, label="Hartree-Fock ($\\theta$=0)")
ax[0].plot(Rs, E_vqe_curve, "o-", ms=3, label="VQE")
ax[0].plot(Rs, E_exact_curve, "k:", lw=2, label="exact (diagonalization)")
ax[0].plot(R_eq, E_min, "r*", ms=14)
ax[0].set_xlabel("R (Angstrom)"); ax[0].set_ylabel("total energy (Ha)")
ax[0].set_title("H$_2$ potential energy curve"); ax[0].legend(fontsize=8)

ax[1].plot(Rs, 1000*(E_hf_curve - E_exact_curve), "s-", ms=3)
ax[1].axhline(1.6, color="r", ls=":", label="chemical accuracy")
ax[1].set_xlabel("R (Angstrom)"); ax[1].set_ylabel("HF error (mHa)")
ax[1].set_title("What Hartree-Fock misses = correlation energy"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Exercise 6 — the punchline of Module 3

Look at the right-hand plot. At $R = 0.74$ Å, Hartree–Fock is off by ~13 mHa. As the bond
**stretches**, that error grows without bound: the two electrons increasingly need to be
on *opposite* atoms, which a single configuration simply cannot describe.

Plot the optimal $\theta$ and the antibonding fraction against $R$ and interpret.

In [ ]:
theta_curve, anti_curve = [], []
for R in Rs:
    Hr, enuc = h2_hamiltonian(R)
    th, e, _ = run_vqe(Hr, enuc, theta0=0.0)
    theta_curve.append(abs(th))
    # TODO: append the probability of |10> for the optimized state
    ...

plt.figure(figsize=(6.2, 3.4))
plt.plot(Rs, 100*np.array(anti_curve), "o-", ms=3)
plt.xlabel("R (Angstrom)"); plt.ylabel("antibonding weight (%)")
plt.tight_layout(); plt.show()

## Deliverables

Submit this notebook with all cells run, plus short written answers to:

1. Your value of $R_{eq}$ and $E_0$ (Ha and eV), compared to the lecture's numbers.
2. The correlation energy at $R_{eq}$ in kcal/mol, and whether Hartree–Fock reaches
   chemical accuracy.
3. Roughly how many energy evaluations COBYLA needed, and how that would scale if the
   ansatz had 30 parameters instead of 1.
4. One sentence on why the noisy VQE sometimes reports an energy *below* the true ground
   state, even though the variational principle says $E(\theta) \ge E_0$ always.
5. **Challenge (optional):** replace the ansatz with a generic two-parameter circuit
   (`ry` on each qubit plus a `cx`) and see whether VQE still finds the right energy.
   What goes wrong, and what does that tell you about ansatz design?

### What is next
**Lab 6** goes back to the full 4-qubit picture — spin orbitals, Jordan–Wigner, UCCSD —
and shows you exactly what the "reduced mapping" bought you.